# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading, exploring, and analyzing the FAIR² Mental Health Survey dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library with Python.

### Dataset Source
The dataset is described using a [Croissant schema](https://mlcommons.org/croissant/), obtained from an online URL. All fields, columns, and record sets in the dataset are referenced by their unique `@id`, ensuring reproducibility and clarity.

In [ ]:
# Ensure the mlcroissant library is installed
!pip install mlcroissant pandas

## 1. Data Loading
Load Croissant dataset metadata and prepare for further exploration.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Title: ", metadata.name)
print("Description: ", metadata.description)

## 2. Data Overview
Review available record sets, their unique `@id`s, and the fields (`cr:Field`) and columns (`cr:column`) contained in each. All references are given using their `@id`.

In [ ]:
# List record sets and their fields/columns, referencing by @id
if hasattr(metadata, 'record_sets'):
    print('Record Sets:')
    for rs in metadata.record_sets:
        print(f"- @id: {rs['@id']}  Name: {rs['name']}")
        if 'fields' in rs:
            print('   Fields (by @id):')
            for field in rs['fields']:
                print(f"      - {field['@id']} : {field.get('name', '')}")
        if 'columns' in rs:
            print('   Columns (by @id):')
            for col in rs['columns']:
                print(f"      - {col['@id']} : {col.get('name', '')}")
else:
    print('No record sets found in the metadata.')

# Alternatively, list all record set @ids for later reference
record_set_ids = []
if hasattr(metadata, 'record_sets'):
    for rs in metadata.record_sets:
        record_set_ids.append(rs['@id'])
print('Record Set @ids:', record_set_ids)

## 3. Data Extraction
Load data from each available record set into a pandas DataFrame.

All entities (record sets, fields, columns) are referenced by their `@id`.

In [ ]:
# Extract data for each record set into a dictionary of DataFrames, using the @id
dataframes = {}
for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded DataFrame for record_set @id={record_set_id}, columns: {df.columns.tolist()}")
        else:
            print(f"No records found for record_set @id={record_set_id}")
    except Exception as e:
        print(f"Error loading records for record_set @id={record_set_id}: {e}")

# Choose a record set for demonstration (take first if available)
if dataframes:
    selected_record_set_id = list(dataframes)[0]
    print(f"\nSample rows from record_set @id={selected_record_set_id}:")
    display(dataframes[selected_record_set_id].head())
else:
    selected_record_set_id = None

## 4. Exploratory Data Analysis (EDA)
Apply basic filtering, normalization, and grouping using fields referenced by their `@id`.

> Replace the field `@id`s as needed for specific analysis.

In [ ]:
# --- Example EDA: Filtering and normalization ---

import numpy as np
if selected_record_set_id:
    df = dataframes[selected_record_set_id]
    print(f"Sample columns: {df.columns.tolist()}")
    
    # Replace with a real numeric field @id (e.g., GAD-7 score)
    # For illustration, pick the first numeric-looking column
    numeric_field_id = None
    for col in df.columns:
        try:
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break
        except Exception:
            continue
    
    if numeric_field_id:
        print(f"Selected numeric field @id: {numeric_field_id}")
        threshold = df[numeric_field_id].quantile(0.7) # For demonstration, use the 70th percentile
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records where {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalize the numeric field
        normalized_col = f"{numeric_field_id}_normalized"
        filtered_df[normalized_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized values for {numeric_field_id}:")
        display(filtered_df[[numeric_field_id, normalized_col]].head())

        # Group by a categorical field
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and df[col].nunique() < 10:
                group_field_id = col
                break

        if group_field_id:
            print(f"Grouping by field @id: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            display(grouped_df)
        else:
            print('No suitable categorical field found for grouping.')
    else:
        print('No numeric field found for EDA.')
else:
    print('No record set available for EDA.')

## 5. Visualization
Visualize the distribution of a numeric field and its grouping by a categorical field

> All field references below are by their `@id` as above. Adapt as needed for your dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if selected_record_set_id and numeric_field_id:
    df = dataframes[selected_record_set_id]
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.show()

## 6. Conclusion

This notebook demonstrated how to:
- Load and parse a Croissant dataset by its schema URL with `mlcroissant`
- Explore entities by their `@id` (record sets, fields, columns)
- Load data into pandas DataFrames and conduct basic EDA with references by `@id`
- Visualize numeric fields and their categorical distributions

### Next Steps
- Use field and column `@id`s for precise, reproducible feature selection
- Extend EDA with correlation, missingness, or domain-specific statistics for mental health epidemiology
- Consult the Croissant metadata for data provenance and annotation context

For further information, see the [mlcroissant documentation](https://mlcommons.org/croissant/) and the dataset record.